In [2]:
import pandas as pd
import os

In [3]:
TRANSLATION_PROMPT = """
Sei un traduttore esperto specializzato in traduzioni dall'inglese all'italiano. Traduci il seguente testo in italiano standard, mantenendo:
1. Il significato esatto del testo originale senza aggiungere o omettere informazioni
2. Il registro linguistico appropriato (formale/informale in base al contesto)
3. Espressioni idiomatiche equivalenti quando necessario
4. La corretta struttura sintattica italiana, evitando calchi dall'inglese
5. La terminologia specifica del dominio cinematografico quando presente
6. La punteggiatura e il formato originali dove appropriato
L'obiettivo è ottenere una traduzione che sembri scritta originariamente in italiano da un madrelingua, rispettando la naturalezza espressiva e la scorrevolezza. Il testo da tradurre è una recensione cinematografica.

Recensione da tradurre: {text}
"""

In [ ]:
def translate_text(text, client, settings):
    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": TRANSLATION_PROMPT.format(text=text)}
            ],
            temperature=0
        )
        logger.info("TRANSLATION: Translation successfully generated")
        translation = completion.choices[0].message.content
        translation = translation.replace('\r\n', ' ').replace('\n', ' ')
        translation = translation.replace('"', '""')
        return translation
    except Exception as e:
        logger.error(f"Error during translation generation: {e}")
        return f"Error during translation generation: {e}"

In [ ]:
def translate_reviews_csv(input_path=None, client=None, settings=None):
   try:
       if input_path is None:
           input_path = "../resources/IMDB Dataset Sampled.csv"

       if not os.path.exists(input_path):
           logger.error(f"The file {input_path} does not exist")
           return {"success": False, "error": f"The file {input_path} does not exist"}

       filename, ext = os.path.splitext(input_path)
       output_path = f"IMDB Dataset Sampled With Translate{ext}"

       logger.info(f"Reading CSV file: {input_path}")
       df = pd.read_csv(input_path)

       required_columns = ["Unnamed: 0", "review", "sentiment", "entities", "json"]

       for col in required_columns:
           if col not in df.columns:
               df[col] = ""

       if 'translate' not in df.columns:
           df['translate'] = ""

       if 'progressive_index' not in df.columns:
           df['progressive_index'] = range(len(df))

       for idx, row in df.iterrows():
           review_text = row['review']

           if pd.isna(review_text) or review_text.strip() == "":
               df.at[idx, 'translate'] = ""
               continue

           translation = translate_text(review_text, client, settings)

           df.at[idx, 'translate'] = translation

           if (idx + 1) % 10 == 0:
               logger.info(f"Processed {idx + 1} reviews out of {len(df)}")

       ordered_columns = ["Unnamed: 0", "review", "sentiment", "entities", "json", "translate", "progressive_index"]
       existing_columns = [col for col in ordered_columns if col in df.columns]
       df = df[existing_columns]

       df.to_csv(output_path, index=False, quoting=1)
       logger.info(f"Translated file saved at: {output_path}")
       return {"success": True, "output_file": output_path}

   except Exception as e:
       error_message = f"Error during CSV file processing: {e}"
       logger.error(error_message)
       return {"success": False, "error": error_message}